<a href="https://colab.research.google.com/github/natasya783/Laporan-Kendaraan/blob/main/Laporan%20Kendaraan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 6. LAPORAN KENDARAAN (Bobot 10% + Bonus Export CSV)
# =====================================================================
def ringkasan_biaya(user_id, id_kendaraan):
    """Menghitung total servis, biaya, rata-rata, dan servis terbanyak."""
    with koneksi_db() as conn:
        cur = conn.execute("""
            SELECT COUNT(*), COALESCE(SUM(biaya), 0), COALESCE(AVG(biaya), 0)
            FROM riwayat_servis WHERE id_kendaraan = ?
        """, (id_kendaraan,))
        total_servis, total_biaya, rata_rata = cur.fetchone()
        cur = conn.execute("""
            SELECT jenis_servis, COUNT(*) AS jumlah
            FROM riwayat_servis
            WHERE id_kendaraan = ?
            GROUP BY jenis_servis
            ORDER BY jumlah DESC, jenis_servis ASC
            LIMIT 1
        """, (id_kendaraan,))
        terbanyak = cur.fetchone()
    return total_servis, total_biaya, rata_rata, terbanyak

def laporan_kendaraan(user_id):
    """Mencetak laporan ringkas kendaraan dan servis."""
    kendaraan = pilih_kendaraan(user_id)
    if not kendaraan:
        return
    total, biaya, rata, terbanyak = ringkasan_biaya(user_id, kendaraan[0])
    with koneksi_db() as conn:
        cur = conn.execute("""
            SELECT jenis_servis, km_target_berikutnya
            FROM riwayat_servis
            WHERE id_kendaraan = ?
            ORDER BY id DESC LIMIT 1
        """, (kendaraan[0],))
        berikutnya = cur.fetchone()

    print("\n==============================")
    print("       CARE MOTOR REPORT")
    print("==============================")
    print(f"Motor          : {kendaraan[2]} {kendaraan[3]}")
    print(f"Nomor Polisi   : {kendaraan[1]}")
    print(f"KM Saat Ini    : {kendaraan[5]:,} Km")
    print(f"Total Servis   : {total} kali")
    print(f"Total Biaya    : Rp{biaya:,.0f}")
    print(f"Biaya Rata-rata: Rp{rata:,.0f}")
    print(f"Servis Terbanyak: {terbanyak[0]} ({terbanyak[1]} kali)" if terbanyak else "Servis Terbanyak: -")
    if berikutnya:
        print(f"Servis Berikutnya: {berikutnya[0]}")
        print(f"Target           : {berikutnya[1]:,} Km")
    else:
        print("Servis Berikutnya: Belum ada data")
    print("==============================")

def export_csv(user_id):
    """Mengekspor riwayat servis kendaraan terpilih ke file CSV."""
    kendaraan = pilih_kendaraan(user_id)
    if not kendaraan:
        return
    nama_file = f"riwayat_servis_{kendaraan[0]}.csv"
    with koneksi_db() as conn:
        cur = conn.execute("""
            SELECT tanggal_servis, jenis_servis, km_saat_servis, biaya,
                   km_target_berikutnya, catatan
            FROM riwayat_servis WHERE id_kendaraan = ?
            ORDER BY id
        """, (kendaraan[0],))
        data = cur.fetchall()
    with open(nama_file, "w", newline="", encoding="utf-8-sig") as file:
        writer = csv.writer(file)
        writer.writerow(["Tanggal", "Jenis Servis", "KM Servis", "Biaya",
                         "Target Berikutnya", "Catatan"])
        writer.writerows(data)
    print(f"Data berhasil diekspor ke {nama_file}.")